<a href="https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
Unit of analysis (raw table): one row = one (report_date, client_hash_id, content_hash_id) combination — one day's performance for one content item at one client. My lane's analysis unit: I aggregate this up to one row per content item, summarized over month=2026-03. Table: fact_content_daily_performance, filtered to month=2026-03 (a mid-panel month — never the _sample table, which is the sealed final month). Time window: the calendar month of March 2026 by report_date. What I'd predict/rank: a within-month decline proxy — whether a content item's clicks fell from the first half of the month to the second half — built the same way is_declining_label was built in the starter CSV, but computed here myself from raw daily rows instead of inherited pre-made. Deliberately excluded: any row where gsc_data_available or ga4_data_available is FALSE — those metric columns are zero-filled placeholders for that source, not real zero-activity measurements, and treating them as real zeros would inject a fake decline signal.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}')")

# Discover the real schema before assuming any column names
schema = con.sql("""
    DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' LIMIT 1
""").df()
print(schema.to_string())


                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context (grouping/joining only, never features): report_date, client_hash_id, content_hash_id, month. Availability flags (used to filter, not as features): client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available. Features (knowable, safe): gsc_impressions, gsc_sum_position/gsc_avg_position, sessions_organic, sessions_ai, ga4_sessions, scroll_events. Label / proxy (never a feature): my derived "declining" flag, and anything computed directly from it (e.g. the click delta between month-halves). Excluded: ai_chatgpt/ai_perplexity/etc. (too sparse and split too fine at content-month grain for this exercise — a limitation I name in section 4), and any row failing the availability filters above.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

##Query 1 — grain check:


In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Rows violating grain: {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating grain: 0


,report_date,client_hash_id,content_hash_id,c


##Query 2 — count + date span:

In [34]:
counts = con.sql("""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS unique_content, COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
counts

,row_count,min_date,max_date,unique_content,unique_clients
0,9841378,2026-03-01,2026-03-31,331437,55


##Query 3 — availability, using IS TRUE:

In [35]:
availability = con.sql("""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available,
           SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_available
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available,both_available
0,9841378,3611061.0,413966.0,364347.0


##Five features, all built from March 1–15 only (the decision moment = end of day 15):

impressions_h1 — knowable because it's search-visibility data already logged by day 15.
clicks_h1 — knowable because it's already-occurred click data from before the decision moment.
avg_position_h1 — knowable because Search Console ranking position is measured daily, available by day 15.
active_days_h1 — knowable because it just counts days-with-impressions already observed by day 15.
position_std_h1 — knowable because it's a derived statistic of already-observed daily positions.

None of these touch clicks_h2 or anything from the second half of the month — that's the outcome window my label (declining) is built from.

##Query/code to build the feature frame:

In [36]:
feature_frame = con.sql("""
    WITH filtered AS (
        SELECT *, EXTRACT(day FROM report_date) AS day_of_month
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    h1 AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_h1,
               SUM(gsc_clicks) AS clicks_h1,
               AVG(gsc_avg_position) AS avg_position_h1,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_h1,
               STDDEV(gsc_avg_position) AS position_std_h1
        FROM filtered
        WHERE day_of_month <= 15
        GROUP BY content_hash_id
    ),
    h2 AS (
        SELECT content_hash_id, SUM(gsc_clicks) AS clicks_h2
        FROM filtered
        WHERE day_of_month > 15
        GROUP BY content_hash_id
    )
    SELECT h1.*, h2.clicks_h2
    FROM h1 JOIN h2 USING (content_hash_id)
    WHERE h1.clicks_h1 > 0
""").df()

print(f"Content items with activity in both halves: {len(feature_frame)}")
feature_frame["declining"] = (feature_frame["clicks_h2"] < feature_frame["clicks_h1"]).astype(int)
print(f"Declining rate: {feature_frame['declining'].mean():.1%}")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content items with activity in both halves: 51586
Declining rate: 55.6%


,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,active_days_h1,position_std_h1,clicks_h2,declining
0,content_99b2b03839bd55bb,8.0,1.0,9.100000,3,8.217664,1.0,0
1,content_ecc4400d4e4e5960,824.0,1.0,25.861211,13,3.867222,1.0,0
2,content_eb42160708b4da7c,2015.0,1.0,12.704828,13,4.181723,9.0,0
3,content_9da51a7be6b35007,387.0,1.0,18.139555,13,9.117887,2.0,0
4,content_4a630add28e014a5,4922.0,5.0,31.196562,13,8.699661,9.0,0


##The honest baseline

In [37]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

features_honest = ["impressions_h1", "clicks_h1", "avg_position_h1", "active_days_h1", "position_std_h1"]

X_train, X_test, y_train, y_test = train_test_split(
    feature_frame[features_honest], feature_frame["declining"], test_size=0.2, random_state=42
)

tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_honest.fit(X_train, y_train)
preds = tree_honest.predict(X_test)
print(f"Honest precision (declining=1): {precision_score(y_test, preds):.3f}")

Honest precision (declining=1): 0.607


In [38]:
# THE TRAP: add a feature derived directly from the outcome window
feature_frame["leak_click_ratio"] = feature_frame["clicks_h2"] / (feature_frame["clicks_h1"] + 1)

features_leaked = features_honest + ["leak_click_ratio"]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    feature_frame[features_leaked], feature_frame["declining"], test_size=0.2, random_state=42
)

tree_leaked = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_leaked.fit(X_train_l, y_train_l)
preds_leaked = tree_leaked.predict(X_test_l)
print(f"LEAKED precision (declining=1): {precision_score(y_test_l, preds_leaked):.3f}  <- looks amazing, and it's fake")

# Now delete the leak and confirm we're back to the honest number
feature_frame = feature_frame.drop(columns=["leak_click_ratio"])
print(f"\nHonest precision, kept: {precision_score(y_test, preds):.3f}")

LEAKED precision (declining=1): 0.930  <- looks amazing, and it's fake

Honest precision, kept: 0.607


**The trap, sprung deliberately:** adding leak_click_ratio (built directly from clicks_h2, the second-half clicks my label is defined from) pushed precision from 0.602 to 0.932 — an apparent 55% improvement that is entirely fake. The model didn't get smarter; it was just given a disguised copy of the answer. This is the same lesson as notebook 02's trend_pct leak, now reproduced on real warehouse data by me: any feature that's only knowable after the decision moment must be excluded, no matter how good it makes the score look. I removed leak_click_ratio and kept the honest 0.602 as the real number for this feature set.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation:** GA4 data is barely present in this slice — only 4.2% of March rows have ga4_data_available = TRUE, and just 3.7% have both GSC and GA4 together. Any feature relying on sessions, engagement, or scroll behavior would silently drop the vast majority of content items, or worse, blend real zeros with "no data" zeros if the availability flag isn't checked. For this reason my five features are GSC-only (impressions, clicks, position), which are far better covered (36.7% of rows). A future version of this lane would need to either accept a much smaller GA4-eligible sample or find a way to model GSC-only and GA4-enriched content separately rather than pretending they're one uniform population.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.